# Positional Encodings — Exercises

| # | Exercise | Difficulty | Key Concepts |
|---|---|---|---|
| 1 | Sinusoidal PE by hand | ★★ | PE formula, d=4, linear offset M_k |
| 2 | Dot-product decay | ★★ | PE(0)·PE(k), offset-only dependence |
| 3 | RoPE 2D | ★★★ | 2D rotation, relative position proof |
| 4 | ALiBi slopes & bias | ★★ | Slope schedule, local vs global heads |
| 5 | RoPE frequency analysis | ★★★ | Wavelengths, dead dimensions, context |
| 6 | Extrapolation comparison | ★★★ | Sinusoidal vs RoPE vs ALiBi beyond n_train |
| 7 | YaRN zones | ★★★★ | High/mid/low freq, scaling factors |
| 8 | Needle-in-haystack | ★★★★ | Context retrieval, RoPE vs ALiBi |

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

## Exercise 1: Sinusoidal PE by Hand

**Difficulty**: ★★

**Goal**: Compute sinusoidal PE vectors manually for d=4 and verify the linear offset property.

Given:
- d = 4, base = 10000
- Positions: pos = 0, 1, 2

**Tasks:**
1. Compute PE(pos) for pos = 0, 1, 2 — four values each: [sin(pos·ω₀), cos(pos·ω₀), sin(pos·ω₁), cos(pos·ω₁)]
2. Verify PE(0) · PE(0) = d/2 (self dot product = 2)
3. Build the 4×4 rotation matrix M₁ such that PE(pos+1) = M₁ · PE(pos)
4. Verify M₁ · PE(0) = PE(1) and M₁ · PE(1) = PE(2)

In [ ]:
# === Exercise 1: Scaffold ===

d = 4
base = 10000

# Frequencies for dimension pairs 0 and 1
omega_0 = 1.0 / (base ** (0 / d))   # pair 0: 2i/d = 0/4 = 0
omega_1 = 1.0 / (base ** (2 / d))   # pair 1: 2i/d = 2/4 = 0.5

print(f"ω₀ = {omega_0}, ω₁ = {omega_1}")

# Task 1: Compute PE vectors
# PE(pos) = [sin(pos*ω₀), cos(pos*ω₀), sin(pos*ω₁), cos(pos*ω₁)]
# TODO: Compute PE(0), PE(1), PE(2)

# Task 2: Verify PE(0)·PE(0) = d/2
# TODO

# Task 3: Build M_1 — block-diagonal rotation matrix
# M_1 = blockdiag(R(ω₀), R(ω₁)) where R(θ) = [[cos(θ), sin(θ)], [-sin(θ), cos(θ)]]
# TODO

# Task 4: Verify M_1 · PE(0) = PE(1)
# TODO

In [ ]:
# === Exercise 1: Solution ===

d = 4
base = 10000

# Frequencies
omega_0 = 1.0 / (base ** (0 / d))    # = 1.0
omega_1 = 1.0 / (base ** (2 / d))    # = 1/100

print(f"=== Frequencies ===")
print(f"ω₀ = 1/10000^(0/4) = {omega_0}")
print(f"ω₁ = 1/10000^(2/4) = 1/10000^0.5 = 1/100 = {omega_1}")

# Task 1: Compute PE vectors
print(f"\n=== Task 1: PE Vectors ===")
pe_vectors = {}
for pos in [0, 1, 2]:
    pe = np.array([
        np.sin(pos * omega_0),
        np.cos(pos * omega_0),
        np.sin(pos * omega_1),
        np.cos(pos * omega_1)
    ])
    pe_vectors[pos] = pe
    print(f"PE({pos}) = [{', '.join(f'{v:.6f}' for v in pe)}]")

# Task 2: Self dot product = d/2
print(f"\n=== Task 2: Self Dot Product ===")
for pos in [0, 1, 2]:
    dot_self = pe_vectors[pos] @ pe_vectors[pos]
    print(f"PE({pos})·PE({pos}) = {dot_self:.6f} (expected: d/2 = {d/2})")
    # Proof: sin²(α) + cos²(α) = 1 for each pair, × 2 pairs = 2

# Task 3: Build M_1
print(f"\n=== Task 3: Rotation Matrix M₁ ===")
M_1 = np.zeros((d, d))
for i, omega in enumerate([omega_0, omega_1]):
    c, s = np.cos(omega), np.sin(omega)
    idx = 2 * i
    M_1[idx, idx] = c
    M_1[idx, idx+1] = s
    M_1[idx+1, idx] = -s
    M_1[idx+1, idx+1] = c

print("M₁ =")
for row in M_1:
    print(f"  [{', '.join(f'{v:>10.6f}' for v in row)}]")

# Task 4: Verify M_1 · PE(pos) = PE(pos+1)
print(f"\n=== Task 4: Verification ===")
for pos in [0, 1]:
    predicted = M_1 @ pe_vectors[pos]
    actual = pe_vectors[pos + 1]
    error = np.linalg.norm(predicted - actual)
    print(f"M₁ · PE({pos}) = [{', '.join(f'{v:.6f}' for v in predicted)}]")
    print(f"PE({pos+1})     = [{', '.join(f'{v:.6f}' for v in actual)}]")
    print(f"Error: {error:.2e} ✓\n")

## Exercise 2: Dot-Product Decay

**Difficulty**: ★★

**Goal**: Compute PE(0)·PE(k) for various offsets k and demonstrate that:
1. The dot product depends only on the offset k, not absolute positions
2. The dot product decays (on average) as offset increases
3. The analytical formula ∑ᵢ cos(k·ωᵢ) matches the numerical result

**Parameters**: d=512, base=10000, offsets k = 0, 1, 5, 10, 50, 100, 200

In [ ]:
# === Exercise 2: Scaffold ===

d = 512
base = 10000

def sinusoidal_pe(n, d, base=10000):
    pos = np.arange(n)[:, None]
    i = np.arange(0, d, 2)[None, :]
    angles = pos / base ** (i / d)
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

# TODO: Generate PE for enough positions
# TODO: Compute PE(0)·PE(k) for k = 0, 1, 5, 10, 50, 100, 200
# TODO: Verify PE(p)·PE(p+k) = PE(0)·PE(k) for multiple base positions p
# TODO: Compute analytical formula: sum_i cos(k * omega_i)
# TODO: Plot decay curve

In [ ]:
# === Exercise 2: Solution ===

d = 512
base = 10000
n = 500

def sinusoidal_pe(n, d, base=10000):
    pos = np.arange(n)[:, None]
    i = np.arange(0, d, 2)[None, :]
    angles = pos / base ** (i / d)
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

pe = sinusoidal_pe(n, d, base)

# Task 1: Dot products at various offsets
print("=== Dot-Product at Various Offsets ===")
offsets = [0, 1, 5, 10, 50, 100, 200]
print(f"{'k':>6} {'PE(0)·PE(k)':>14} {'PE(10)·PE(10+k)':>18} {'PE(50)·PE(50+k)':>18} {'Offset-only?':>14}")
print("-" * 75)
for k in offsets:
    d1 = pe[0] @ pe[k]
    d2 = pe[10] @ pe[10 + k]
    d3 = pe[50] @ pe[50 + k]
    match = np.isclose(d1, d2, atol=1e-8) and np.isclose(d1, d3, atol=1e-8)
    print(f"{k:>6} {d1:>14.4f} {d2:>18.4f} {d3:>18.4f} {'✓' if match else '✗':>14}")

# Task 2: Analytical formula
print(f"\n=== Analytical Verification ===")
dim_pairs = np.arange(d // 2)
omegas = 1.0 / (base ** (2 * dim_pairs / d))
print(f"{'k':>6} {'Computed':>12} {'Analytical':>12} {'Error':>12}")
print("-" * 46)
for k in offsets:
    computed = pe[0] @ pe[k]
    analytical = np.sum(np.cos(k * omegas))
    err = abs(computed - analytical)
    print(f"{k:>6} {computed:>12.4f} {analytical:>12.4f} {err:>12.2e}")

# Task 3: Plot
if HAS_MPL:
    all_offsets = np.arange(n)
    all_dots = np.array([pe[0] @ pe[k] for k in all_offsets])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(all_offsets, all_dots, 'b-', alpha=0.7, linewidth=0.5)
    axes[0].axhline(y=0, color='gray', ls='--', alpha=0.3)
    axes[0].scatter(offsets, [pe[0] @ pe[k] for k in offsets], c='red', s=50, zorder=5)
    axes[0].set_xlabel('Offset k')
    axes[0].set_ylabel('PE(0) · PE(k)')
    axes[0].set_title(f'Dot-Product Decay (d={d})')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(all_offsets[:60], all_dots[:60], 'b-o', markersize=2, alpha=0.7)
    axes[1].axhline(y=0, color='gray', ls='--', alpha=0.3)
    axes[1].set_xlabel('Offset k')
    axes[1].set_ylabel('PE(0) · PE(k)')
    axes[1].set_title('Zoom: first 60 positions')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\n=== Key Insight ===")
print(f"PE(0)·PE(0) = {pe[0] @ pe[0]:.1f} = d/2 = {d//2} (maximum)")
print(f"Dot product oscillates and decays with offset → sinusoidal PE implicitly encodes distance")

## Exercise 3: RoPE 2D

**Difficulty**: ★★★

**Goal**: For d=2 with θ = π/4, compute rotated query vectors at positions 0,1,2,3 and verify that the dot product q̃ᵢ·k̃ⱼ depends only on i−j.

**Given**:
- d = 2, θ = π/4
- q = (1, 0), k = (0, 1)

**Tasks:**
1. Build 2×2 rotation matrices R₀, R₁, R₂, R₃
2. Compute R_m · q for m = 0, 1, 2, 3
3. Compute R_n · k for n = 0, 1, 2, 3  
4. Build 4×4 dot-product table: dot(R_m q, R_n k) for all (m,n)
5. Verify: entries with same m−n have same value

In [ ]:
# === Exercise 3: Scaffold ===

theta = np.pi / 4
q = np.array([1.0, 0.0])
k = np.array([0.0, 1.0])

def rotation_2d(angle):
    """Build 2×2 rotation matrix."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

# TODO: Build R_0, R_1, R_2, R_3
# TODO: Compute rotated q and k at each position
# TODO: Build dot-product table
# TODO: Verify same offset → same dot product

In [ ]:
# === Exercise 3: Solution ===

theta = np.pi / 4  # 45 degrees
q = np.array([1.0, 0.0])
k = np.array([0.0, 1.0])
n_pos = 4

def rotation_2d(angle):
    """Build 2×2 rotation matrix for given angle."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

# Step 1: Build rotation matrices R_0, R_1, R_2, R_3
print("=== Step 1: Rotation Matrices ===\n")
R = []
for m in range(n_pos):
    R_m = rotation_2d(m * theta)
    R.append(R_m)
    print(f"R_{m} (angle = {m}·π/4 = {m*45}°):")
    print(f"  [{R_m[0,0]:+.4f}  {R_m[0,1]:+.4f}]")
    print(f"  [{R_m[1,0]:+.4f}  {R_m[1,1]:+.4f}]\n")

# Step 2: Compute rotated queries and keys at each position
print("=== Step 2: Rotated Vectors ===\n")
q_rot = []
k_rot = []
for m in range(n_pos):
    q_m = R[m] @ q
    k_m = R[m] @ k
    q_rot.append(q_m)
    k_rot.append(k_m)
    print(f"Position {m}: q̃_{m} = R_{m}·q = ({q_m[0]:+.4f}, {q_m[1]:+.4f}),  "
          f"k̃_{m} = R_{m}·k = ({k_m[0]:+.4f}, {k_m[1]:+.4f})")

# Step 3: Build dot-product table
print("\n=== Step 3: Dot-Product Table ===\n")
dot_table = np.zeros((n_pos, n_pos))
for m in range(n_pos):
    for n in range(n_pos):
        dot_table[m, n] = q_rot[m] @ k_rot[n]

# Print header
header = "q\\k  " + "  ".join(f"  k_{n}  " for n in range(n_pos))
print(header)
print("-" * len(header))
for m in range(n_pos):
    row = f"q_{m}  " + "  ".join(f"{dot_table[m,n]:+.4f}" for n in range(n_pos))
    print(row)

# Step 4: Verify same offset → same dot product
print("\n=== Step 4: Relative Position Property ===\n")
print("Offset → Dot-product values:")
for offset in range(-(n_pos-1), n_pos):
    values = []
    for m in range(n_pos):
        n = m + offset
        if 0 <= n < n_pos:
            values.append(dot_table[m, n])
    if values:
        all_same = np.allclose(values, values[0])
        vals_str = ", ".join(f"{v:+.4f}" for v in values)
        print(f"  offset {offset:+d}: [{vals_str}]  {'✓ all equal' if all_same else '✗ MISMATCH'}")

# Step 5: Analytical verification: (R_m q)^T (R_n k) = q^T R_{n-m} k
print("\n=== Step 5: Analytical Verification ===\n")
print("Verify: (R_m q)ᵀ (R_n k) = qᵀ R_{n-m} k\n")
for m in range(n_pos):
    for n in range(n_pos):
        lhs = q_rot[m] @ k_rot[n]
        R_diff = rotation_2d((n - m) * theta)
        rhs = q @ R_diff @ k
        match = np.isclose(lhs, rhs)
        if m < 2 and n < 3:  # print subset
            print(f"  m={m}, n={n}: LHS={lhs:+.4f}, RHS={rhs:+.4f}  {'✓' if match else '✗'}")

print("\nAll (m,n) pairs match:", np.allclose(
    [[q_rot[m] @ k_rot[n] for n in range(n_pos)] for m in range(n_pos)],
    [[q @ rotation_2d((n-m)*theta) @ k for n in range(n_pos)] for m in range(n_pos)]
))

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: Rotated vectors in 2D
    ax = axes[0]
    colors_q = plt.cm.Blues(np.linspace(0.4, 0.9, n_pos))
    colors_k = plt.cm.Reds(np.linspace(0.4, 0.9, n_pos))
    for m in range(n_pos):
        ax.arrow(0, 0, q_rot[m][0]*0.9, q_rot[m][1]*0.9,
                head_width=0.05, color=colors_q[m], label=f'q̃_{m}' if m < 4 else '')
        ax.arrow(0, 0, k_rot[m][0]*0.9, k_rot[m][1]*0.9,
                head_width=0.05, color=colors_k[m], linestyle='--', label=f'k̃_{m}' if m < 4 else '')
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower left')
    ax.set_title('Rotated q and k vectors')

    # Panel 2: Dot-product matrix
    ax = axes[1]
    im = ax.imshow(dot_table, cmap='RdBu_r', vmin=-1, vmax=1)
    for i in range(n_pos):
        for j in range(n_pos):
            ax.text(j, i, f'{dot_table[i,j]:.2f}', ha='center', va='center', fontsize=10)
    ax.set_xticks(range(n_pos)); ax.set_xticklabels([f'k_{i}' for i in range(n_pos)])
    ax.set_yticks(range(n_pos)); ax.set_yticklabels([f'q_{i}' for i in range(n_pos)])
    ax.set_title('Dot-product table (q̃ᵢᵀ k̃ⱼ)')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Panel 3: Dot product vs offset
    ax = axes[2]
    offsets = list(range(-(n_pos-1), n_pos))
    avg_dots = []
    for off in offsets:
        vals = [dot_table[m, m+off] for m in range(n_pos) if 0 <= m+off < n_pos]
        avg_dots.append(np.mean(vals))
    ax.bar(offsets, avg_dots, color='steelblue', alpha=0.7, edgecolor='navy')
    ax.set_xlabel('Relative offset (n - m)')
    ax.set_ylabel('Dot product')
    ax.set_title('Dot product depends only on offset')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('ex3_rope_2d.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ex3_rope_2d.png")

## Exercise 4: ALiBi Slopes and Bias Matrix

**Difficulty:** ★★☆ Intermediate

ALiBi (Attention with Linear Biases) replaces positional embeddings with a linear bias added directly to attention scores. Each head $h$ gets a geometric slope $m_h$.

**Slope schedule** for $H$ heads:

$$m_h = 2^{-8h/H}, \quad h = 1, 2, \ldots, H$$

**Bias matrix:**

$$\text{bias}[i, j] = -m_h \cdot |i - j|$$

### Tasks

1. **Compute slopes** for $H = 8$ heads. Print as a table with both fraction and decimal.
2. **Build the 12×12 bias matrix** for head $h = 1$ (steepest slope). Print it.
3. **Build the 12×12 bias matrix** for head $h = 8$ (shallowest slope). Print it.
4. **Convert bias to attention weights** using softmax. Compare the effective attention window for head 1 vs head 8.
5. **Identify** which heads act as "local" attention (most mass within ±2) and which act as "global" attention.

In [ ]:
# === Exercise 4: Scaffold ===

H = 8   # number of heads
n = 12  # sequence length

def alibi_slopes(H):
    """Compute ALiBi slopes for H heads."""
    # TODO: m_h = 2^(-8h/H) for h = 1,...,H
    pass

def alibi_bias(slope, n):
    """Build n×n bias matrix for a given slope."""
    # TODO: bias[i,j] = -slope * |i - j|
    pass

# TODO: Print slope table
# TODO: Build bias for steepest and shallowest heads
# TODO: Convert to attention weights via softmax
# TODO: Classify local vs global heads

In [ ]:
# === Exercise 4: Solution ===

H = 8
n = 12

def alibi_slopes(H):
    """Compute ALiBi geometric slope schedule for H heads."""
    return np.array([2**(-8 * h / H) for h in range(1, H + 1)])

def alibi_bias(slope, n):
    """Build n×n causal bias matrix: bias[i,j] = -slope * |i-j|."""
    positions = np.arange(n)
    return -slope * np.abs(positions[:, None] - positions[None, :])

# Step 1: Compute and display slopes
slopes = alibi_slopes(H)
print("=== Step 1: ALiBi Slopes (H=8) ===\n")
print(f"{'Head':<6} {'Exponent':<12} {'Fraction':<16} {'Decimal':<12}")
print("-" * 48)
for h in range(H):
    exp = -8 * (h + 1) / H
    frac_num = 1
    frac_den = int(2 ** (8 * (h + 1) / H))
    print(f"h={h+1:<4} 2^({exp:+.1f})     1/{frac_den:<12}  {slopes[h]:.6f}")

# Step 2: Bias matrix for steepest head (h=1, slope=1/2)
print(f"\n=== Step 2: Bias Matrix — Head 1 (slope = {slopes[0]:.4f}) ===\n")
bias_steep = alibi_bias(slopes[0], n)
print("     " + "  ".join(f"  j={j:<2}" for j in range(n)))
for i in range(n):
    row = f"i={i:<2} " + "  ".join(f"{bias_steep[i,j]:+6.2f}" for j in range(n))
    print(row)

# Step 3: Bias matrix for shallowest head (h=8, slope=1/256)
print(f"\n=== Step 3: Bias Matrix — Head 8 (slope = {slopes[-1]:.6f}) ===\n")
bias_shallow = alibi_bias(slopes[-1], n)
print("     " + "  ".join(f"  j={j:<2}" for j in range(n)))
for i in range(n):
    row = f"i={i:<2} " + "  ".join(f"{bias_shallow[i,j]:+6.3f}" for j in range(n))
    print(row)

# Step 4: Convert to attention weights via softmax
print("\n=== Step 4: Attention Weight Distribution ===\n")
# Use a uniform Q·K score (all zeros) + bias to isolate PE effect
uniform_scores = np.zeros((n, n))

# Causal mask
causal_mask = np.triu(np.ones((n, n)) * (-1e9), k=1)

print(f"{'Head':<6} {'Slope':<10} {'Mass ±2':<10} {'Mass ±4':<10} {'Entropy':<10} {'Type':<8}")
print("-" * 56)

for h in range(H):
    bias_h = alibi_bias(slopes[h], n)
    scores_h = uniform_scores + bias_h + causal_mask
    # Softmax per row
    weights = np.zeros_like(scores_h)
    for i in range(n):
        row = scores_h[i, :i+1]
        exp_row = np.exp(row - np.max(row))
        weights[i, :i+1] = exp_row / exp_row.sum()

    # Compute mass within ±2 and ±4 of diagonal (average over rows with enough context)
    mass_2 = 0
    mass_4 = 0
    count = 0
    for i in range(4, n):  # rows with at least 4 tokens of context
        for j in range(i + 1):
            if abs(i - j) <= 2:
                mass_2 += weights[i, j]
            if abs(i - j) <= 4:
                mass_4 += weights[i, j]
        count += 1
    mass_2 /= count
    mass_4 /= count

    # Entropy of last row
    w_last = weights[-1, :n]
    w_last = w_last[w_last > 0]
    entropy = -np.sum(w_last * np.log2(w_last))

    head_type = "LOCAL" if mass_2 > 0.8 else ("MIXED" if mass_2 > 0.3 else "GLOBAL")
    print(f"h={h+1:<4} {slopes[h]:<10.6f} {mass_2:<10.4f} {mass_4:<10.4f} {entropy:<10.4f} {head_type}")

# Step 5: Classification summary
print("\n=== Step 5: Head Classification ===\n")
for h in range(H):
    bias_h = alibi_bias(slopes[h], n)
    scores_h = uniform_scores + bias_h + causal_mask
    weights = np.zeros_like(scores_h)
    for i in range(n):
        row = scores_h[i, :i+1]
        exp_row = np.exp(row - np.max(row))
        weights[i, :i+1] = exp_row / exp_row.sum()
    mass_2 = 0
    count = 0
    for i in range(4, n):
        for j in range(i + 1):
            if abs(i - j) <= 2:
                mass_2 += weights[i, j]
        count += 1
    mass_2 /= count
    category = "LOCAL (sharp decay)" if mass_2 > 0.8 else ("MIXED" if mass_2 > 0.3 else "GLOBAL (gentle decay)")
    print(f"  Head {h+1}: slope={slopes[h]:.6f} → {category}")

print("\nKey insight: Steeper slopes (small h) → local attention.")
print("Shallower slopes (large h) → global/uniform attention.")
print("This gives ALiBi a multi-scale inductive bias without learning.")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel 1: Bias matrices comparison
    ax = axes[0]
    im = ax.imshow(bias_steep, cmap='RdBu_r', aspect='auto')
    ax.set_title(f'Head 1 bias (slope={slopes[0]:.2f})')
    ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Panel 2: Attention weights for steepest vs shallowest
    ax = axes[1]
    # Last row attention
    bias_s = alibi_bias(slopes[0], n) + causal_mask
    bias_g = alibi_bias(slopes[-1], n) + causal_mask
    def get_weights(scores):
        row = scores[-1]
        exp_row = np.exp(row - np.max(row))
        return exp_row / exp_row.sum()
    w_steep = get_weights(bias_s)
    w_shallow = get_weights(bias_g)
    x = np.arange(n)
    ax.bar(x - 0.15, w_steep, 0.3, label=f'Head 1 (slope={slopes[0]:.2f})', color='coral', alpha=0.8)
    ax.bar(x + 0.15, w_shallow, 0.3, label=f'Head 8 (slope={slopes[-1]:.4f})', color='steelblue', alpha=0.8)
    ax.set_xlabel('Key position'); ax.set_ylabel('Attention weight')
    ax.set_title('Last query: local vs global head')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Slope schedule
    ax = axes[2]
    ax.bar(range(1, H+1), slopes, color='darkgreen', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Head index'); ax.set_ylabel('Slope value')
    ax.set_title('ALiBi slope schedule (geometric)')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('ex4_alibi.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ex4_alibi.png")

## Exercise 5: RoPE Frequency Analysis

**Difficulty:** ★★★ Advanced

RoPE assigns each dimension pair $i$ a rotation frequency:

$$\theta_i = \text{base}^{-2i/d}$$

where $\text{base} = 10000$ (default). The wavelength (positions for one full rotation) is:

$$\lambda_i = \frac{2\pi}{\theta_i} = 2\pi \cdot \text{base}^{2i/d}$$

### Tasks

1. **Compute all frequencies** $\theta_i$ and wavelengths $\lambda_i$ for $d = 128$, $\text{base} = 10000$.
2. **Print a table** showing pair index, frequency, wavelength, and number of full rotations within training context $L = 4096$.
3. **Identify "dead dimensions"**: pairs where $\lambda_i > L$ (less than 1 full rotation during training). Count them.
4. **Compute the effective resolution**: what is the minimum offset $\Delta$ that produces at least a 1° rotation in the fastest pair?
5. **Compare bases**: repeat for $\text{base} = 500000$ (Llama 3.1) and show how the frequency spectrum shifts.

In [ ]:
# === Exercise 5: Scaffold ===

d = 128
base = 10000
L_train = 4096  # training context length

# TODO: Compute theta_i for all pairs i = 0, ..., d/2 - 1
# TODO: Compute wavelength lambda_i = 2*pi / theta_i
# TODO: Compute rotations_in_train = L_train / lambda_i
# TODO: Identify dead dimensions (rotations_in_train < 1)
# TODO: Compute minimum offset for 1° rotation
# TODO: Repeat with base = 500000

In [ ]:
# === Exercise 5: Solution ===

d = 128
base_default = 10000
L_train = 4096
n_pairs = d // 2

# Step 1: Compute frequencies and wavelengths
print("=== Step 1: RoPE Frequencies (d=128, base=10000) ===\n")
pair_indices = np.arange(n_pairs)
theta = base_default ** (-2.0 * pair_indices / d)
wavelength = 2 * np.pi / theta
rotations = L_train / wavelength

# Step 2: Print table (sampled)
print(f"{'Pair i':<8} {'θ_i':<14} {'λ_i (pos)':<14} {'Rotations in L=4096':<20} {'Status'}")
print("-" * 72)
sample_indices = [0, 1, 2, 3, 4, 8, 16, 24, 32, 40, 48, 56, 60, 62, 63]
for i in sample_indices:
    status = "ACTIVE" if rotations[i] >= 1 else "DEAD (<1 rot)"
    print(f"{i:<8} {theta[i]:<14.6e} {wavelength[i]:<14.2f} {rotations[i]:<20.4f} {status}")

# Step 3: Dead dimensions
n_dead = np.sum(rotations < 1)
first_dead = np.argmax(rotations < 1) if n_dead > 0 else n_pairs
print(f"\n=== Step 3: Dead Dimensions ===")
print(f"Total pairs: {n_pairs}")
print(f"Active pairs (≥1 rotation): {n_pairs - n_dead}")
print(f"Dead pairs (<1 rotation):   {n_dead}")
if n_dead > 0:
    print(f"First dead pair: i = {first_dead}")
    print(f"  θ_{first_dead} = {theta[first_dead]:.6e}")
    print(f"  λ_{first_dead} = {wavelength[first_dead]:.2f} positions")
    print(f"  Rotations in L={L_train}: {rotations[first_dead]:.4f}")

# Step 4: Minimum offset for 1° rotation
print(f"\n=== Step 4: Effective Resolution ===")
one_degree_rad = np.pi / 180
# Fastest pair is i=0
theta_fastest = theta[0]
min_offset = one_degree_rad / theta_fastest
print(f"Fastest frequency: θ_0 = {theta_fastest:.6f} rad/pos")
print(f"1° = {one_degree_rad:.6f} rad")
print(f"Minimum offset for 1° rotation: {min_offset:.6f} positions")
print(f"→ Even adjacent tokens rotate by {np.degrees(theta_fastest):.2f}° in the fastest pair")

# Slowest active pair
if first_dead > 0:
    theta_slowest = theta[first_dead - 1]
    min_offset_slow = one_degree_rad / theta_slowest
    print(f"\nSlowest active pair (i={first_dead-1}): θ = {theta_slowest:.6e} rad/pos")
    print(f"Minimum offset for 1° rotation: {min_offset_slow:.2f} positions")

# Step 5: Compare with base = 500000 (Llama 3.1)
print(f"\n=== Step 5: Base Comparison ===\n")
bases = [10000, 500000]
print(f"{'Pair':<8}", end="")
for b in bases:
    print(f"  {'base='+str(b):<22}", end="")
print()
print("-" * 54)

for i in sample_indices[:10]:
    print(f"i={i:<5}", end="")
    for b in bases:
        t = b ** (-2.0 * i / d)
        lam = 2 * np.pi / t
        rot = L_train / lam
        print(f"  λ={lam:>10.1f} rot={rot:>6.3f}", end="")
    print()

# Dead dimension comparison
print(f"\n{'Metric':<30} {'base=10000':<15} {'base=500000':<15}")
print("-" * 60)
for b in bases:
    t_b = b ** (-2.0 * pair_indices / d)
    lam_b = 2 * np.pi / t_b
    rot_b = L_train / lam_b
    n_dead_b = int(np.sum(rot_b < 1))
    first_dead_b = int(np.argmax(rot_b < 1)) if n_dead_b > 0 else n_pairs
    if b == bases[0]:
        print(f"{'Dead dimensions':<30} {n_dead_b:<15} ", end="")
    else:
        print(f"{n_dead_b}")
    if b == bases[0]:
        print(f"{'First dead pair':<30} {first_dead_b:<15} ", end="")
    else:
        print(f"{first_dead_b}")
    if b == bases[0]:
        print(f"{'Max wavelength (active)':<30} {wavelength[first_dead_b-1]:<15.1f} ", end="")
    else:
        lam_b_active = lam_b[first_dead_b - 1] if first_dead_b > 0 and first_dead_b < n_pairs else lam_b[-1]
        print(f"{lam_b_active:.1f}")

print("\nKey insight: Larger base stretches wavelengths → more dead dimensions")
print("but also enables encoding longer-range dependencies in active dimensions.")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel 1: Wavelength spectrum
    ax = axes[0]
    for b, color, label in [(10000, 'steelblue', 'base=10000'), (500000, 'coral', 'base=500000')]:
        t = b ** (-2.0 * pair_indices / d)
        lam = 2 * np.pi / t
        ax.semilogy(pair_indices, lam, color=color, label=label, linewidth=2)
    ax.axhline(y=L_train, color='red', linestyle='--', alpha=0.7, label=f'L_train={L_train}')
    ax.set_xlabel('Dimension pair index')
    ax.set_ylabel('Wavelength (positions)')
    ax.set_title('Wavelength Spectrum')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 2: Rotations in training
    ax = axes[1]
    ax.bar(pair_indices, np.minimum(rotations, 100), color=['steelblue' if r >= 1 else 'lightcoral' for r in rotations],
           alpha=0.7, edgecolor='none')
    ax.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='1 full rotation')
    ax.set_xlabel('Dimension pair index')
    ax.set_ylabel('Full rotations in L=4096')
    ax.set_title('Rotations During Training (base=10000)')
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Frequency comparison
    ax = axes[2]
    for b, color, label in [(10000, 'steelblue', 'base=10000'), (500000, 'coral', 'base=500000')]:
        t = b ** (-2.0 * pair_indices / d)
        ax.semilogy(pair_indices, t, color=color, label=label, linewidth=2)
    ax.set_xlabel('Dimension pair index')
    ax.set_ylabel('Frequency θ_i (rad/pos)')
    ax.set_title('Frequency Spectrum')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('ex5_rope_freq.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ex5_rope_freq.png")

## Exercise 6: Extrapolation Comparison

**Difficulty:** ★★★ Advanced

A critical question for positional encodings: **what happens when you use positions beyond the training range?**

Given training length $L_{\text{train}} = 512$, compare sinusoidal, RoPE, and ALiBi at:
- $1\times$ (within training: $n = 512$)
- $2\times$ (mild extrapolation: $n = 1024$)
- $4\times$ (heavy extrapolation: $n = 2048$)

### Tasks

1. **Sinusoidal**: Compute dot products $\text{PE}(m)^T \text{PE}(n)$ for positions beyond $L_{\text{train}}$. Do they stay bounded?
2. **RoPE**: Compute position-pair dot products $(R_m q)^T (R_n k)$ at extrapolated positions. Does the relative-position property still hold?
3. **ALiBi**: Compute bias values at extrapolated distances. How does the linear penalty behave?
4. **Quantify degradation**: For each method, measure how the dot-product pattern at $4\times$ differs from the pattern at $1\times$.
5. **Plot** the dot-product-vs-offset curve for each method at each scale.

In [ ]:
# === Exercise 6: Scaffold ===

d = 64
L_train = 512
scales = [1, 2, 4]  # 1x, 2x, 4x

def sinusoidal_pe(pos, d):
    """Build sinusoidal PE vector for given position."""
    # TODO
    pass

def rope_dot(m, n, d, base=10000):
    """Compute RoPE dot product between positions m and n."""
    # TODO
    pass

def alibi_penalty(m, n, slope):
    """Compute ALiBi penalty for positions m and n."""
    # TODO
    pass

# TODO: For each method and each scale, compute dot products at various offsets
# TODO: Quantify degradation
# TODO: Visualize

In [ ]:
# === Exercise 6: Solution ===

d = 64
L_train = 512
base = 10000
np.random.seed(42)

def sinusoidal_pe(pos, d):
    """Compute sinusoidal PE vector at position pos."""
    pe = np.zeros(d)
    for i in range(d // 2):
        omega = 1.0 / (10000 ** (2 * i / d))
        pe[2 * i] = np.sin(omega * pos)
        pe[2 * i + 1] = np.cos(omega * pos)
    return pe

def rope_dot_product(m, n, q, k, d, base=10000):
    """Compute (R_m q)^T (R_n k) for RoPE."""
    result = 0.0
    for i in range(d // 2):
        theta_i = base ** (-2.0 * i / d)
        angle_m = m * theta_i
        angle_n = n * theta_i
        # Rotate q pair
        q0, q1 = q[2*i], q[2*i+1]
        qr0 = q0 * np.cos(angle_m) - q1 * np.sin(angle_m)
        qr1 = q0 * np.sin(angle_m) + q1 * np.cos(angle_m)
        # Rotate k pair
        k0, k1 = k[2*i], k[2*i+1]
        kr0 = k0 * np.cos(angle_n) - k1 * np.sin(angle_n)
        kr1 = k0 * np.sin(angle_n) + k1 * np.cos(angle_n)
        result += qr0 * kr0 + qr1 * kr1
    return result

# Fixed q and k vectors for RoPE
q = np.random.randn(d) * 0.1
k = np.random.randn(d) * 0.1

# ALiBi slope (single head)
alibi_slope = 2**(-8.0/8)  # head 1 of 8

scales = [1, 2, 4]
max_offset = 128  # offsets to measure
offsets = np.arange(0, max_offset)

# Step 1: Sinusoidal extrapolation
print("=== Step 1: Sinusoidal PE — Extrapolation ===\n")
print("Sinusoidal PE uses fixed frequencies → values always bounded in [-1, 1]")
print("But: the model never saw these position combinations during training.\n")

sin_dots = {}
for scale in scales:
    L = L_train * scale
    anchor = L - 1  # query at end of sequence
    dots = []
    for off in offsets:
        pos_k = anchor - off
        if pos_k >= 0:
            pe_q = sinusoidal_pe(anchor, d)
            pe_k = sinusoidal_pe(pos_k, d)
            dots.append(pe_q @ pe_k)
        else:
            dots.append(np.nan)
    sin_dots[scale] = np.array(dots)
    print(f"  {scale}x (L={L}): anchor={anchor}, dot[offset=0]={dots[0]:.4f}, "
          f"dot[offset=10]={dots[10]:.4f}, range=[{np.nanmin(dots):.4f}, {np.nanmax(dots):.4f}]")

# Step 2: RoPE extrapolation
print(f"\n=== Step 2: RoPE — Extrapolation ===\n")
print("RoPE rotates q,k → dot product depends only on offset (by construction).")
print("But: high-frequency pairs wrap unpredictably at unseen positions.\n")

rope_dots = {}
for scale in scales:
    L = L_train * scale
    anchor = L - 1
    dots = []
    for off in offsets:
        pos_k = anchor - off
        if pos_k >= 0:
            dots.append(rope_dot_product(anchor, pos_k, q, k, d, base))
        else:
            dots.append(np.nan)
    rope_dots[scale] = np.array(dots)
    print(f"  {scale}x (L={L}): dot[offset=0]={dots[0]:.4f}, "
          f"dot[offset=10]={dots[10]:.4f}, dot[offset=50]={dots[50]:.4f}")

# Verify relative-position property still holds at extrapolated positions
print("\n  Relative-position check at 4x:")
L4 = L_train * 4
for off in [1, 5, 10]:
    # Check from two different anchor points
    d1 = rope_dot_product(L4 - 1, L4 - 1 - off, q, k, d, base)
    d2 = rope_dot_product(L4 // 2, L4 // 2 - off, q, k, d, base)
    print(f"    offset={off}: anchor={L4-1} → {d1:.6f}, anchor={L4//2} → {d2:.6f}, "
          f"match={'✓' if np.isclose(d1, d2, atol=1e-10) else '✗'}")

# Step 3: ALiBi extrapolation
print(f"\n=== Step 3: ALiBi — Extrapolation ===\n")
print("ALiBi: bias = -slope × |offset|. Linear penalty extends naturally.\n")

alibi_dots = {}
for scale in scales:
    L = L_train * scale
    penalties = -alibi_slope * offsets.astype(float)
    alibi_dots[scale] = penalties
    print(f"  {scale}x (L={L}): bias[offset=0]={penalties[0]:.4f}, "
          f"bias[offset=10]={penalties[10]:.4f}, bias[offset=100]={penalties[100]:.4f}")

print("\nALiBi extrapolates perfectly: the linear function is defined for all distances.")

# Step 4: Quantify degradation
print(f"\n=== Step 4: Degradation Quantification ===\n")
print("Metric: Mean absolute difference in dot-product pattern vs 1x baseline\n")

ref_offsets = min(max_offset, L_train)  # compare over shared offsets
print(f"{'Method':<14} {'1x→2x MAD':<14} {'1x→4x MAD':<14} {'Notes'}")
print("-" * 60)

# Sinusoidal
sin_mad_2 = np.nanmean(np.abs(sin_dots[2][:ref_offsets] - sin_dots[1][:ref_offsets]))
sin_mad_4 = np.nanmean(np.abs(sin_dots[4][:ref_offsets] - sin_dots[1][:ref_offsets]))
print(f"{'Sinusoidal':<14} {sin_mad_2:<14.6f} {sin_mad_4:<14.6f} {'Pattern shifts with anchor'}")

# RoPE
rope_mad_2 = np.nanmean(np.abs(rope_dots[2][:ref_offsets] - rope_dots[1][:ref_offsets]))
rope_mad_4 = np.nanmean(np.abs(rope_dots[4][:ref_offsets] - rope_dots[1][:ref_offsets]))
print(f"{'RoPE':<14} {rope_mad_2:<14.6f} {rope_mad_4:<14.6f} {'Offset-only → zero drift'}")

# ALiBi
ali_mad_2 = np.nanmean(np.abs(alibi_dots[2][:ref_offsets] - alibi_dots[1][:ref_offsets]))
ali_mad_4 = np.nanmean(np.abs(alibi_dots[4][:ref_offsets] - alibi_dots[1][:ref_offsets]))
print(f"{'ALiBi':<14} {ali_mad_2:<14.6f} {ali_mad_4:<14.6f} {'Exact extrapolation'}")

print("\nKey insight:")
print("  • Sinusoidal: dot products change because the absolute position changes")
print("  • RoPE: dot products depend only on offset → zero degradation at same offsets")
print("  • ALiBi: linear bias is defined for all distances → perfect extrapolation")
print("  However, RoPE encounters new rotation angles at extrapolated offsets that")
print("  the model never trained on, causing attention pattern breakdown.")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    colors = {1: 'steelblue', 2: 'orange', 4: 'red'}

    # Panel 1: Sinusoidal
    ax = axes[0]
    for scale in scales:
        valid = ~np.isnan(sin_dots[scale])
        ax.plot(offsets[valid], sin_dots[scale][valid], color=colors[scale],
                label=f'{scale}x (L={L_train*scale})', alpha=0.8, linewidth=1.5)
    ax.set_xlabel('Offset from query')
    ax.set_ylabel('PE dot product')
    ax.set_title('Sinusoidal: dot product shifts')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 2: RoPE
    ax = axes[1]
    for scale in scales:
        valid = ~np.isnan(rope_dots[scale])
        ax.plot(offsets[valid], rope_dots[scale][valid], color=colors[scale],
                label=f'{scale}x (L={L_train*scale})', alpha=0.8, linewidth=1.5)
    ax.set_xlabel('Offset from query')
    ax.set_ylabel('q·k dot product')
    ax.set_title('RoPE: offset-invariant')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: ALiBi
    ax = axes[2]
    for scale in scales:
        ax.plot(offsets, alibi_dots[scale], color=colors[scale],
                label=f'{scale}x (L={L_train*scale})', alpha=0.8, linewidth=2)
    ax.set_xlabel('Offset from query')
    ax.set_ylabel('Attention bias')
    ax.set_title('ALiBi: perfect extrapolation')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('ex6_extrapolation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ex6_extrapolation.png")

## Exercise 7: YaRN Frequency Zones

**Difficulty:** ★★★ Advanced

YaRN (Yet another RoPE extensioN) classifies each RoPE dimension pair into three zones based on how many rotations it completes during training:

| Zone | Condition | Treatment |
|------|-----------|-----------|
| High-frequency | $\lambda_i < L_{\text{train}} / \beta_{\text{fast}}$ | No scaling (keep original) |
| Low-frequency | $\lambda_i > L_{\text{train}} / \beta_{\text{slow}}$ | Full scaling by factor $s$ |
| Medium | Between | Smooth interpolation via ramp $\gamma(i)$ |

The ramp function:

$$\gamma(i) = \frac{\lambda_i / L_{\text{train}} - \beta_{\text{slow}}}{\beta_{\text{fast}} - \beta_{\text{slow}}}$$

The scaled frequency:

$$\theta'_i = \theta_i \cdot \left(\frac{1 - \gamma(i)}{s} + \gamma(i)\right)$$

### Tasks

1. Compute all $\theta_i$, $\lambda_i$ for $d = 64$, $\text{base} = 10000$.
2. For $s = 4$ (extending $4\times$), $\beta_{\text{fast}} = 32$, $\beta_{\text{slow}} = 1$: classify every dimension pair.
3. Compute $\gamma(i)$ for medium-frequency pairs. Show the smooth ramp.
4. Compute the scaled frequencies $\theta'_i$ and verify they form a smooth spectrum.
5. Compare original vs scaled wavelengths and verify high-freq pairs are untouched.

In [ ]:
# === Exercise 7: Scaffold ===

d = 64
base = 10000
L_train = 2048
s = 4          # scaling factor (4x extension)
beta_fast = 32
beta_slow = 1

# TODO: Compute theta_i and lambda_i for all pairs
# TODO: Classify each pair as high/medium/low frequency
# TODO: Compute gamma ramp for medium-frequency pairs
# TODO: Compute scaled frequencies theta'_i
# TODO: Compare original vs scaled wavelengths

In [ ]:
# === Exercise 7: Solution ===

d = 64
base = 10000
L_train = 2048
s = 4           # extension factor: 2048 → 8192
beta_fast = 32  # high-freq boundary
beta_slow = 1   # low-freq boundary
n_pairs = d // 2

# Step 1: Compute frequencies and wavelengths
pair_idx = np.arange(n_pairs)
theta = base ** (-2.0 * pair_idx / d)
wavelength = 2 * np.pi / theta

print("=== Step 1: Original RoPE Frequencies (d=64, base=10000) ===\n")
print(f"{'Pair':<6} {'θ_i':<14} {'λ_i':<14} {'Rotations/L':<14}")
print("-" * 50)
for i in [0, 1, 2, 4, 8, 12, 16, 20, 24, 28, 31]:
    rot = L_train / wavelength[i]
    print(f"{i:<6} {theta[i]:<14.6e} {wavelength[i]:<14.2f} {rot:<14.4f}")

# Step 2: Classify each pair
print(f"\n=== Step 2: Zone Classification ===")
print(f"Boundaries: λ_high < L/β_fast = {L_train}/{beta_fast} = {L_train/beta_fast:.1f}")
print(f"            λ_low  > L/β_slow = {L_train}/{beta_slow} = {L_train/beta_slow:.1f}\n")

lambda_high = L_train / beta_fast
lambda_low = L_train / beta_slow

zones = []
gammas = np.zeros(n_pairs)
for i in range(n_pairs):
    if wavelength[i] < lambda_high:
        zones.append("HIGH")
        gammas[i] = 1.0  # keep original
    elif wavelength[i] > lambda_low:
        zones.append("LOW")
        gammas[i] = 0.0  # full scaling
    else:
        zones.append("MEDIUM")
        # Ramp
        gammas[i] = (wavelength[i] / L_train - beta_slow) / (beta_fast - beta_slow)
        gammas[i] = np.clip(gammas[i], 0, 1)

print(f"{'Pair':<6} {'λ_i':<12} {'Zone':<10} {'γ(i)':<10} {'Treatment'}")
print("-" * 56)
for i in range(n_pairs):
    treatment = {
        "HIGH": "No scaling (preserve)",
        "LOW": f"Scale by 1/{s}",
        "MEDIUM": f"Blend: {gammas[i]:.4f}"
    }[zones[i]]
    if i < 20 or zones[i] != zones[min(i+1, n_pairs-1)] or i == n_pairs - 1:
        print(f"{i:<6} {wavelength[i]:<12.2f} {zones[i]:<10} {gammas[i]:<10.4f} {treatment}")

n_high = zones.count("HIGH")
n_medium = zones.count("MEDIUM")
n_low = zones.count("LOW")
print(f"\nCounts: HIGH={n_high}, MEDIUM={n_medium}, LOW={n_low}")

# Step 3: Gamma ramp visualization
print(f"\n=== Step 3: Gamma Ramp ===\n")
print("γ(i) interpolates smoothly between 0 (full scale) and 1 (no scale)")
print("in the medium-frequency zone.\n")
medium_pairs = [i for i in range(n_pairs) if zones[i] == "MEDIUM"]
if medium_pairs:
    print(f"Medium-frequency pairs: {medium_pairs}")
    for i in medium_pairs:
        bar = "█" * int(gammas[i] * 30) + "░" * int((1 - gammas[i]) * 30)
        print(f"  pair {i:>2}: γ={gammas[i]:.4f} [{bar}]")

# Step 4: Compute scaled frequencies
print(f"\n=== Step 4: Scaled Frequencies ===\n")
theta_scaled = theta * ((1 - gammas) / s + gammas)
wavelength_scaled = 2 * np.pi / theta_scaled

print(f"{'Pair':<6} {'Zone':<8} {'θ_orig':<14} {'θ_scaled':<14} {'λ_orig':<12} {'λ_scaled':<12} {'Ratio'}")
print("-" * 80)
for i in range(n_pairs):
    ratio = wavelength_scaled[i] / wavelength[i]
    if i < 16 or zones[i] != zones[min(i+1, n_pairs-1)] or i == n_pairs - 1:
        print(f"{i:<6} {zones[i]:<8} {theta[i]:<14.6e} {theta_scaled[i]:<14.6e} "
              f"{wavelength[i]:<12.2f} {wavelength_scaled[i]:<12.2f} {ratio:<.4f}")

# Step 5: Verify properties
print(f"\n=== Step 5: Verification ===\n")
print("High-frequency pairs should be untouched:")
for i in range(min(n_high, 5)):
    match = np.isclose(theta[i], theta_scaled[i])
    print(f"  pair {i}: θ_orig={theta[i]:.6e}, θ_scaled={theta_scaled[i]:.6e} → {'✓ unchanged' if match else '✗ CHANGED'}")

print(f"\nLow-frequency pairs should be scaled by 1/{s}:")
low_pairs = [i for i in range(n_pairs) if zones[i] == "LOW"]
for i in low_pairs[:5]:
    expected = theta[i] / s
    match = np.isclose(theta_scaled[i], expected)
    print(f"  pair {i}: θ_orig/s={expected:.6e}, θ_scaled={theta_scaled[i]:.6e} → {'✓ matches' if match else '✗ MISMATCH'}")

print(f"\nOriginal max context: {L_train}")
print(f"Scaled max context: {L_train * s}")
print(f"Low-freq wavelength scaling: {s}x → covers {s}x longer context")

# Effective context check
max_active_wavelength = np.max(wavelength_scaled[wavelength_scaled < L_train * s])
print(f"\nMax active scaled wavelength: {max_active_wavelength:.1f}")
print(f"Target context: {L_train * s}")
print(f"Coverage: {'✓ sufficient' if max_active_wavelength <= L_train * s else '✗ insufficient'}")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Panel 1: Zone classification
    ax = axes[0]
    zone_colors = {'HIGH': 'green', 'MEDIUM': 'orange', 'LOW': 'red'}
    colors = [zone_colors[z] for z in zones]
    ax.bar(pair_idx, wavelength, color=colors, alpha=0.7, edgecolor='none')
    ax.axhline(y=lambda_high, color='green', linestyle='--', label=f'λ_high={lambda_high:.0f}', alpha=0.8)
    ax.axhline(y=lambda_low, color='red', linestyle='--', label=f'λ_low={lambda_low:.0f}', alpha=0.8)
    ax.set_xlabel('Dimension pair index')
    ax.set_ylabel('Wavelength (positions)')
    ax.set_title('Zone Classification')
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 2: Gamma ramp
    ax = axes[1]
    ax.plot(pair_idx, gammas, 'o-', color='darkorange', markersize=4, linewidth=2)
    ax.fill_between(pair_idx, gammas, alpha=0.2, color='orange')
    ax.set_xlabel('Dimension pair index')
    ax.set_ylabel('γ(i)')
    ax.set_title('YaRN Ramp Function')
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    # Mark zones
    if n_high > 0:
        ax.axvspan(0, n_high - 0.5, alpha=0.1, color='green', label='High-freq')
    if n_low > 0:
        first_low = n_pairs - n_low
        ax.axvspan(first_low - 0.5, n_pairs - 0.5, alpha=0.1, color='red', label='Low-freq')
    ax.legend(fontsize=8, loc='center left')

    # Panel 3: Original vs scaled wavelengths
    ax = axes[2]
    ax.semilogy(pair_idx, wavelength, 'o-', color='steelblue', label='Original', markersize=3)
    ax.semilogy(pair_idx, wavelength_scaled, 's-', color='coral', label='YaRN scaled', markersize=3)
    ax.axhline(y=L_train, color='blue', linestyle=':', alpha=0.5, label=f'L_train={L_train}')
    ax.axhline(y=L_train * s, color='red', linestyle=':', alpha=0.5, label=f'L_target={L_train*s}')
    ax.set_xlabel('Dimension pair index')
    ax.set_ylabel('Wavelength (positions)')
    ax.set_title('Original vs YaRN-Scaled Wavelengths')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('ex7_yarn_zones.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ex7_yarn_zones.png")

## Exercise 8: Needle-in-a-Haystack Attention Simulation

**Difficulty:** ★★★ Advanced

Simulate the "needle-in-a-haystack" test: can the model attend to a specific token (the "needle") buried at various depths in a long sequence?

We model this as: query $q$ at the end of a sequence tries to attend to a matching key $k_{\text{needle}}$ placed at position $p$, surrounded by random distractor keys $k_{\text{noise}}$.

### Setup
- $d = 64$, $n_{\text{train}} = 512$
- Query $q$ and needle key $k_{\text{needle}}$ have high dot-product similarity
- Noise keys $k_{\text{noise}}$ have lower similarity
- Test at context lengths: $512, 1024, 2048, 4096$
- Test needle positions: start (10%), middle (50%), end (90%)

### Tasks

1. Build the attention score matrix for each PE method (RoPE, ALiBi, no PE).
2. Compute the attention weight on the needle token at each (depth, context-length) pair.
3. Build a 2D heatmap showing needle retrieval probability across depth × context length.
4. Identify which PE method degrades least at long contexts and deep needle positions.

In [ ]:
# === Exercise 8: Scaffold ===

d = 64
n_train = 512
context_lengths = [512, 1024, 2048, 4096]
needle_depths = [0.1, 0.25, 0.5, 0.75, 0.9]  # fraction from start

np.random.seed(42)

# TODO: Generate q, k_needle (similar), and k_noise (random)
# TODO: For each PE method: compute attention scores with PE
# TODO: For each (context_length, needle_depth): measure attention on needle
# TODO: Build heatmap
# TODO: Compare methods

In [ ]:
# === Exercise 8: Solution ===

d = 64
n_train = 512
base = 10000
context_lengths = [512, 1024, 2048, 4096]
needle_depths = [0.1, 0.25, 0.5, 0.75, 0.9]
np.random.seed(42)

# --- Helper functions ---
def apply_rope_pair(vec, pos, d, base=10000):
    """Apply RoPE rotation to a single vector at position pos."""
    result = vec.copy()
    for i in range(d // 2):
        theta_i = base ** (-2.0 * i / d)
        angle = pos * theta_i
        c, s = np.cos(angle), np.sin(angle)
        v0, v1 = result[2*i], result[2*i+1]
        result[2*i]   = v0 * c - v1 * s
        result[2*i+1] = v0 * s + v1 * c
    return result

def stable_softmax(x):
    """Numerically stable softmax."""
    e = np.exp(x - np.max(x))
    return e / e.sum()

# --- Generate vectors ---
# Query at end of sequence
q = np.random.randn(d) * 0.3

# Needle key: similar to query (high dot product)
k_needle = q + np.random.randn(d) * 0.05  # close to q

# Noise keys: random (low expected dot product with q)
max_ctx = max(context_lengths)
k_noise_pool = np.random.randn(max_ctx, d) * 0.3

print("=== Setup ===")
print(f"q · k_needle (no PE) = {q @ k_needle:.4f}")
print(f"q · k_noise (avg)    = {np.mean([q @ k_noise_pool[j] for j in range(100)]):.4f}")
print(f"d={d}, base={base}, n_train={n_train}\n")

# --- Run needle-in-haystack for each method ---
methods = ['No PE', 'RoPE', 'ALiBi (h=4/8)']
alibi_slope = 2**(-8 * 4 / 8)  # middle head (h=4 of 8)

results = {m: np.zeros((len(context_lengths), len(needle_depths))) for m in methods}

print("=== Running Needle-in-a-Haystack ===\n")

for ci, ctx_len in enumerate(context_lengths):
    for di, depth in enumerate(needle_depths):
        needle_pos = int(depth * (ctx_len - 1))
        query_pos = ctx_len - 1

        # Build key matrix: noise everywhere, needle at needle_pos
        keys = k_noise_pool[:ctx_len].copy()
        keys[needle_pos] = k_needle.copy()

        # --- Method 1: No PE ---
        scores_nope = np.array([q @ keys[j] for j in range(ctx_len)])
        weights_nope = stable_softmax(scores_nope)
        results['No PE'][ci, di] = weights_nope[needle_pos]

        # --- Method 2: RoPE ---
        q_rope = apply_rope_pair(q, query_pos, d, base)
        scores_rope = np.array([
            q_rope @ apply_rope_pair(keys[j], j, d, base)
            for j in range(ctx_len)
        ])
        weights_rope = stable_softmax(scores_rope)
        results['RoPE'][ci, di] = weights_rope[needle_pos]

        # --- Method 3: ALiBi ---
        scores_alibi = np.array([q @ keys[j] for j in range(ctx_len)])
        bias = np.array([-alibi_slope * abs(query_pos - j) for j in range(ctx_len)])
        scores_alibi = scores_alibi + bias
        weights_alibi = stable_softmax(scores_alibi)
        results['ALiBi (h=4/8)'][ci, di] = weights_alibi[needle_pos]

# Print results table
for method in methods:
    print(f"--- {method} ---")
    print(f"{'Context':<10}", end="")
    for depth in needle_depths:
        print(f"  depth={depth:<5}", end="")
    print()
    print("-" * 60)
    for ci, ctx_len in enumerate(context_lengths):
        extrap = "  ◀ train" if ctx_len <= n_train else f"  ({ctx_len//n_train}× extrap)"
        print(f"n={ctx_len:<6}", end="")
        for di in range(len(needle_depths)):
            w = results[method][ci, di]
            print(f"  {w:<10.6f}", end="")
        print(extrap)
    print()

# Analysis
print("=== Analysis ===\n")
for method in methods:
    mat = results[method]
    # Degradation: compare 4096 vs 512
    avg_512 = np.mean(mat[0, :])
    avg_4096 = np.mean(mat[-1, :])
    degradation = (avg_512 - avg_4096) / avg_512 * 100 if avg_512 > 0 else 0

    # Middle vs edge at longest context
    mid_weight = mat[-1, 2]  # depth=0.5 at ctx=4096
    edge_weight = (mat[-1, 0] + mat[-1, -1]) / 2  # avg of start and end

    print(f"{method}:")
    print(f"  Avg needle weight at 512:  {avg_512:.6f}")
    print(f"  Avg needle weight at 4096: {avg_4096:.6f}")
    print(f"  Degradation: {degradation:.1f}%")
    print(f"  Middle vs edge at 4096: mid={mid_weight:.6f}, edge={edge_weight:.6f}")
    print()

print("Key insights:")
print("• No PE: attention weight dilutes uniformly as context grows (1/n effect)")
print("• RoPE: nearby tokens get boosted → middle-of-context needle harder to find")
print("• ALiBi: strong recency bias → early needles get heavily penalized")
print("• The 'lost in the middle' effect is most visible with distance-based PEs")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    for idx, method in enumerate(methods):
        ax = axes[idx]
        mat = results[method]
        im = ax.imshow(mat, cmap='YlOrRd', aspect='auto',
                      vmin=0, vmax=max(0.01, np.max(mat)))
        ax.set_xticks(range(len(needle_depths)))
        ax.set_xticklabels([f'{d:.0%}' for d in needle_depths])
        ax.set_yticks(range(len(context_lengths)))
        ax.set_yticklabels(context_lengths)
        ax.set_xlabel('Needle depth (from start)')
        ax.set_ylabel('Context length')
        ax.set_title(f'{method}')

        # Annotate cells
        for i in range(len(context_lengths)):
            for j in range(len(needle_depths)):
                val = mat[i, j]
                color = 'white' if val > np.max(mat) * 0.6 else 'black'
                ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                       fontsize=7, color=color)

        plt.colorbar(im, ax=ax, shrink=0.8, label='Attention weight')

    plt.suptitle('Needle-in-a-Haystack: Attention on Target Token', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('ex8_needle.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ex8_needle.png")